In [ ]:
import scanpy as sc
import pandas as pd

from itertools import combinations

from tqdm import tqdm
import random

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed


In [ ]:
panel = pd.read_csv('/Users/mathieu/Library/CloudStorage/GoogleDrive-mbourdenx@me.com/My Drive/UCL/Projects/MERSCOPEvsXENIUM/Panel/panel.csv')

In [ ]:
panel.index = panel.Gene

In [ ]:
adata = sc.read_h5ad('../../Reference_MTG_RNAseq_final-nuclei.2022-06-07.h5ad')

In [ ]:
missing = list()

for i in panel.Gene:
    if i not in adata.var.index:
        print(i)
        missing.append(i)

In [ ]:
# Remove missing genes
panel = panel.drop(index=missing)

In [ ]:
pairs = list(combinations(panel.Gene, 2))
len(pairs)

In [ ]:
# Random sampling of all possible pairs
mecr = {}

for i in raandom.sample(pairs, 1000):

    gene1, gene2 = i[0], i[1]
    n_both = (adata[:, gene1].X.todense() > 0) & (adata[:, gene2].X.todense() > 0)
    n_one = (adata[:, gene1].X.todense() > 0) | (adata[:, gene2].X.todense() > 0)

    mecr[i] = n_both.sum() / n_one.sum()

In [ ]:
def compute_mecr(pair, adata):

    gene1, gene2 = pair[0], pair[1]
    n_both = (adata[:, gene1].X.todense() > 0) & (adata[:, gene2].X.todense() > 0)
    n_one = (adata[:, gene1].X.todense() > 0) | (adata[:, gene2].X.todense() > 0)

    return [pair, n_both.sum() / n_one.sum()]


In [ ]:
# Parallel processing of all possible combinations
mecr = Parallel(n_jobs=20, verbose=1)(delayed(compute_mecr)(pair, adata) for pair in pairs)

In [ ]:
mecr_dict = dict()

for i in mecr:
    mecr_dict[i[0]] = i[1]

In [ ]:
plt.figure(figsize=(4,3), constrained_layout=True)
plt.hist(mecr_dict.values(), bins=100)
plt.axvline(x=0.025, c='r', linestyle='dashed')
plt.ylabel('Counts')
plt.xlabel('MECR')
plt.savefig('../Figures/mecr_snrnaseq.png', dpi=300)
plt.show()

In [ ]:
selected_pairs = []

for i,j in zip(mecr_dict.keys(), mecr_dict.values()):
    if j < 0.025:
        selected_pairs.append(i)
    

In [ ]:
len(selected_pairs)

In [ ]:
np.save(file='../../selected_pairs.npy', arr=selected_pairs)

In [ ]:
pairs = np.load('../../selected_pairs.npy')

xen_mecr = []

for i in tqdm(pairs):
    gene1, gene2 = i[0], i[1]

    if gene1 in list(xen_7513_adata.var.index):
        if gene2 in list(xen_7513_adata.var.index):
            n_both = (xen_7513_adata[:, gene1].X.todense() > 0) & (xen_7513_adata[:, gene2].X.todense() >0)
            n_one = (xen_7513_adata[:, gene1].X.todense() > 0) | (xen_7513_adata[:, gene2].X.todense() >0)
            xen_mecr.append(n_both.sum() / n_one.sum())

mer_mecr = []

for i in tqdm(pairs):
    gene1, gene2 = i[0], i[1]
    mer_n_both = (mer_7513_adata[:, gene1].X.todense() > 0) & (mer_7513_adata[:, gene2].X.todense() >0)
    mer_n_one = (mer_7513_adata[:, gene1].X.todense() > 0) | (mer_7513_adata[:, gene2].X.todense() >0)
    mer_mecr.append(mer_n_both.sum() / mer_n_one.sum())

In [ ]:
gene1 = 'SLC17A7'
gene2 = 'GAD1'


fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(6,3), constrained_layout=True)

mer_n_both = (mer_7513_adata[:, gene1].X.todense() > 0) & (mer_7513_adata[:, gene2].X.todense() >0)
mer_n_one = (mer_7513_adata[:, gene1].X.todense() > 0) | (mer_7513_adata[:, gene2].X.todense() >0)
mer_mecr = mer_n_both.sum() / mer_n_one.sum()

ax[0].plot(mer_7513_adata[:, gene1].X.todense(), mer_7513_adata[:, gene2].X.todense(), '.')
ax[0].set_title(f'MERSCOPE - MECR: {mer_mecr:.2f}')
ax[0].set_xlabel(gene1)
ax[0].set_ylabel(gene2)
ax[0].set_ylim(-5,140)
ax[0].set_xlim(-10,250)

xen_n_both = (xen_7513_adata[:, gene1].X.todense() > 0) & (xen_7513_adata[:, gene2].X.todense() >0)
xen_n_one = (xen_7513_adata[:, gene1].X.todense() > 0) | (xen_7513_adata[:, gene2].X.todense() >0)
xen_mecr = xen_n_both.sum() / xen_n_one.sum()

ax[1].plot(xen_7513_adata[:, gene1].X.todense(), xen_7513_adata[:, gene2].X.todense(), '.')
ax[1].set_title(f'Xenium - MECR: {xen_mecr:.2f}')
ax[1].set_xlabel(gene1)
ax[1].set_ylabel(gene2)
ax[1].set_ylim(-5,120)
ax[1].set_xlim(-10,140)

fig.savefig(os.path.join(figure_path, f'{gene1}-{gene2}.png'), dpi=300)

plt.show()